In [1]:
from dataclasses import dataclass, field
from typing import Dict, Any
import pandas as pd


@dataclass
class TimeSeriesBundle:
    data: pd.DataFrame
    meta: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        """
        Initialize metadata after object creation.
        """
        if not isinstance(self.data.index, pd.DatetimeIndex):
            raise ValueError("DataFrame index must be a DatetimeIndex.")

        # Ensure sorted index
        self.data = self.data.sort_index()

        # Initialize metadata
        self.meta = {
            "frequency": pd.infer_freq(self.data.index),
            "index_start": self.data.index.min(),
            "index_end": self.data.index.max(),
            "n_rows": len(self.data),
            "n_columns": self.data.shape[1],
            "columns": self._build_column_metadata()
        }

    def _build_column_metadata(self) -> Dict[str, Dict[str, Any]]:
        """
        Build per-column metadata.
        """
        col_meta = {}

        for col in self.data.columns:
            series = self.data[col]
            col_meta[col] = {
                "first_valid": series.first_valid_index(),
                "last_valid": series.last_valid_index(),
                "n_valid": series.notna().sum(),
                "n_missing": series.isna().sum(),
            }

        return col_meta

    def add_series(self, name: str, series: pd.Series, align: bool = True):
        """
        Add a new time series to the bundle.

        Parameters
        ----------
        name : str
            Column name to assign.
        series : pd.Series
            Series with DatetimeIndex.
        align : bool
            If True, align to existing index.
        """
        if not isinstance(series.index, pd.DatetimeIndex):
            raise ValueError("Series index must be a DatetimeIndex.")

        if align:
            series = series.reindex(self.data.index)

        self.data[name] = series

        # Update metadata
        self.meta["columns"][name] = {
            "first_valid": series.first_valid_index(),
            "last_valid": series.last_valid_index(),
            "n_valid": series.notna().sum(),
            "n_missing": series.isna().sum(),
        }

        self.meta["n_columns"] = self.data.shape[1]

    def refresh_metadata(self):
        """
        Recalculate all metadata (useful after bulk edits).
        """
        self.meta["frequency"] = pd.infer_freq(self.data.index)
        self.meta["index_start"] = self.data.index.min()
        self.meta["index_end"] = self.data.index.max()
        self.meta["n_rows"] = len(self.data)
        self.meta["n_columns"] = self.data.shape[1]
        self.meta["columns"] = self._build_column_metadata()

In [2]:
import pandas as pd
from pathlib import Path
from typing import Union, List, Optional, Literal

AggMethod = Literal["mean", "sum", "median", "min", "max", "first", "last"]


def _floor_to_month_start(idx: pd.DatetimeIndex) -> pd.DatetimeIndex:
    # Works for any timestamp within a month (and preserves timezone if present)
    return idx - pd.to_timedelta(idx.day - 1, unit="D")


def _calendar_floor_index(idx: pd.DatetimeIndex, freq: str) -> pd.DatetimeIndex:
    """
    Calendar 'floor' for non-fixed freqs. Uses safe datetime math for MS,
    and Period conversion for others.
    """
    f = freq.upper()

    if f == "MS":
        return _floor_to_month_start(idx)

    # For other calendar freqs, Period-based conversion is usually fine.
    mapping = {
        "M":  ("M", "M"),    # month end
        "QS": ("Q", "QS"),   # quarter start
        "Q":  ("Q", "Q"),    # quarter end
        "AS": ("Y", "YS"),   # year start (legacy alias)
        "YS": ("Y", "YS"),   # year start
        "A":  ("Y", "Y"),    # year end (legacy alias)
        "Y":  ("Y", "Y"),    # year end
    }

    if f not in mapping:
        raise ValueError(
            f"Non-fixed floor frequency '{freq}' not recognized. "
            f"Use one of: {sorted(['MS', *mapping.keys()])}, or a fixed freq like 'D', 'H', '15min'."
        )

    period_freq, ts_freq = mapping[f]
    return idx.to_period(period_freq).to_timestamp(ts_freq)


def load_timeseries_csv(
    path: Union[str, Path],
    time_col: str,
    value_cols: Union[str, List[str]],
    floor_freq: Optional[str] = None,
    dropna_time: bool = True,
    floor_agg: Optional[AggMethod] = None,          # required if flooring creates duplicates
    resample_freq: Optional[str] = None,
    resample_how: Optional[AggMethod] = None,       # required if resample_freq is set
) -> pd.DataFrame:
    """
    Load and clean a CSV time series file.

    - Converts `time_col` to datetime and sets as DatetimeIndex
    - Optionally floors timestamps (fixed freqs via .floor; calendar freqs via helpers)
    - Optionally resamples to a target frequency

    Safety rule:
    - If flooring creates duplicate timestamps, you MUST specify how to aggregate
      (either floor_agg, or resample_freq + resample_how).
    """

    path = Path(path).expanduser()
    df = pd.read_csv(path)

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    if dropna_time:
        df = df.dropna(subset=[time_col])

    df = df.set_index(time_col).sort_index()

    if isinstance(value_cols, str):
        value_cols = [value_cols]

    df = df[value_cols].apply(pd.to_numeric, errors="coerce")

    # ---------- Flooring (labeling / alignment) ----------
    if floor_freq is not None:
        idx = df.index

        # Fixed freqs use .floor(); calendar freqs use helper
        try:
            floored = idx.floor(floor_freq)
        except ValueError as e:
            # includes "non-fixed frequency" and some "period frequency" errors
            floored = _calendar_floor_index(idx, floor_freq)

        # If flooring collapses multiple rows into same timestamp, require explicit aggregation choice
        if floored.has_duplicates:
            if floor_agg is None and (resample_freq is None or resample_how is None):
                raise ValueError(
                    f"Flooring to '{floor_freq}' would create duplicate timestamps (multiple observations per bin). "
                    f"To avoid silent aggregation, you must either:\n"
                    f"  1) specify floor_agg='mean'|'sum'|'median'|'min'|'max'|'first'|'last', OR\n"
                    f"  2) specify resample_freq=... AND resample_how=... to aggregate via resampling.\n"
                    f"Example (discharge): resample_freq='{floor_freq}', resample_how='mean'"
                )

            df = df.copy()
            df.index = floored

            if floor_agg is not None:
                gb = df.groupby(df.index)
                df = getattr(gb, floor_agg)()
        else:
            df = df.copy()
            df.index = floored

    # ---------- Resampling (true binning/aggregation) ----------
    if resample_freq is not None:
        if resample_how is None:
            raise ValueError(
                "You set resample_freq but did not set resample_how. "
                "Specify resample_how='mean'|'sum'|'median'|'min'|'max'|'first'|'last'."
            )
        rs = df.resample(resample_freq)
        df = getattr(rs, resample_how)()

    return df

In [3]:
testdf = load_timeseries_csv('~/signal-extraction/vae_input_data/AR_cfs.csv','time','cfs','MS')

In [4]:
testdf

,cfs
time,
1940-01-01,24300.0
1940-02-01,57500.0
1940-03-01,69600.0
1940-04-01,29800.0
1940-05-01,11300.0
...,...
2025-02-01,7730.0
2025-03-01,6400.0
2025-04-01,7480.0


In [5]:
print(testdf.head())
print(type(testdf.index))
print(testdf.index.freq)   # may be None; that's okay

                cfs
time               
1940-01-01  24300.0
1940-02-01  57500.0
1940-03-01  69600.0
1940-04-01  29800.0
1940-05-01  11300.0
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>
None


In [6]:
flow_series = testdf["cfs"]
type(flow_series)
# pandas.Series

pandas.core.series.Series

In [7]:
bundle = TimeSeriesBundle(testdf)

In [8]:
print(type(bundle.data.index))      # should be pandas.DatetimeIndex
print(bundle.data.head())
print(bundle.meta["frequency"])

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>
                cfs
time               
1940-01-01  24300.0
1940-02-01  57500.0
1940-03-01  69600.0
1940-04-01  29800.0
1940-05-01  11300.0
MS


In [9]:
series1 = load_timeseries_csv('~/signal-extraction/vae_input_data/psl_nino34_index_clean.csv','DATE','INDEX','MS')
series1

,INDEX
DATE,
1870-01-01,-1.00
1870-02-01,-1.20
1870-03-01,-0.83
1870-04-01,-0.81
1870-05-01,-1.27
...,...
2025-01-01,-0.76
2025-02-01,-0.39
2025-03-01,0.05


In [10]:
nino = series1['INDEX']
type(nino)

pandas.core.series.Series

In [11]:
bundle.add_series('INDEX',nino, align = True)

In [12]:
bundle.refresh_metadata()

In [13]:
print(bundle.data.columns)

Index(['cfs', 'INDEX'], dtype='object')


In [14]:
print(bundle.meta["columns"]['INDEX'])

{'first_valid': Timestamp('1940-01-01 00:00:00'), 'last_valid': Timestamp('2025-05-01 00:00:00'), 'n_valid': np.int64(1025), 'n_missing': np.int64(1)}


In [15]:
print(bundle)

TimeSeriesBundle(data=                cfs  INDEX
time                      
1940-01-01  24300.0   1.18
1940-02-01  57500.0   1.38
1940-03-01  69600.0   1.19
1940-04-01  29800.0   0.80
1940-05-01  11300.0   0.34
...             ...    ...
2025-02-01   7730.0  -0.39
2025-03-01   6400.0   0.05
2025-04-01   7480.0  -0.08
2025-05-01   4090.0  -0.08
2025-06-01   2500.0    NaN

[1026 rows x 2 columns], meta={'frequency': 'MS', 'index_start': Timestamp('1940-01-01 00:00:00'), 'index_end': Timestamp('2025-06-01 00:00:00'), 'n_rows': 1026, 'n_columns': 2, 'columns': {'cfs': {'first_valid': Timestamp('1940-01-01 00:00:00'), 'last_valid': Timestamp('2025-06-01 00:00:00'), 'n_valid': np.int64(1026), 'n_missing': np.int64(0)}, 'INDEX': {'first_valid': Timestamp('1940-01-01 00:00:00'), 'last_valid': Timestamp('2025-05-01 00:00:00'), 'n_valid': np.int64(1025), 'n_missing': np.int64(1)}}})


In [16]:
bundle2 = TimeSeriesBundle(nino)

IndexError: tuple index out of range

In [18]:
import pycwt

ModuleNotFoundError: No module named 'pycwt'

In [19]:
%pip install pycwt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 7.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [24]:
import pycwt as wlt
import pandas as pd
import numpy as np

In [5]:
dc = pd.read_csv("~/signal-extraction/vae_input_data/AR_cfs.csv")
dc['time'] = pd.to_datetime(dc['time'])

dc = dc.set_index('time')
dc

,cfs
time,
1940-01-31,24300.0
1940-02-29,57500.0
1940-03-31,69600.0
1940-04-30,29800.0
1940-05-31,11300.0
...,...
2025-02-28,7730.0
2025-03-31,6400.0
2025-04-30,7480.0


In [8]:
print(pycwt.__version__)

0.4.0b1.dev10+g3343016af.d20251028


In [11]:
import tqdm

In [13]:
dir(pycwt)

['DOG',
 'MexicanHat',
 'Morlet',
 'Paul',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_version_',
 'absolute_import',
 'ar1',
 'ar1_spectrum',
 'chi2',
 'cwt',
 'division',
 'fft',
 'fft_kwargs',
 'find',
 'get_cache_dir',
 'helpers',
 'icwt',
 'mothers',
 'numpy',
 'print_function',
 'rednoise',
 'significance',
 'tqdm',
 'unicode_literals',
 'version',
 'wavelet',
 'wct',
 'wct_significance',
 'xwt']

In [22]:
help(pycwt.wavelet)

Help on module pycwt.wavelet in pycwt:

NAME
    pycwt.wavelet - PyCWT core wavelet transform functions.

FUNCTIONS
    cwt(signal, dt, dj=0.08333333333333333, s0=-1, J=-1, wavelet='morlet', freqs=None)
        Continuous wavelet transform of the signal at specified scales.

        Parameters
        ----------
        signal : numpy.ndarray, list
            Input signal array.
        dt : float
            Sampling interval.
        dj : float, optional
            Spacing between discrete scales. Default value is 1/12.
            Smaller values will result in better scale resolution, but
            slower calculation and plot.
        s0 : float, optional
            Smallest scale of the wavelet. Default value is 2*dt.
        J : float, optional
            Number of scales less one. Scales range from s0 up to
            s0 * 2**(J * dj), which gives a total of (J + 1) scales.
            Default is J = (log2(N * dt / s0)) / dj.
        wavelet : instance of Wavelet class, or

In [20]:
help(pycwt.cwt)

Help on function cwt in module pycwt.wavelet:

cwt(signal, dt, dj=0.08333333333333333, s0=-1, J=-1, wavelet='morlet', freqs=None)
    Continuous wavelet transform of the signal at specified scales.

    Parameters
    ----------
    signal : numpy.ndarray, list
        Input signal array.
    dt : float
        Sampling interval.
    dj : float, optional
        Spacing between discrete scales. Default value is 1/12.
        Smaller values will result in better scale resolution, but
        slower calculation and plot.
    s0 : float, optional
        Smallest scale of the wavelet. Default value is 2*dt.
    J : float, optional
        Number of scales less one. Scales range from s0 up to
        s0 * 2**(J * dj), which gives a total of (J + 1) scales.
        Default is J = (log2(N * dt / s0)) / dj.
    wavelet : instance of Wavelet class, or string
        Mother wavelet class. Default is Morlet wavelet.
    freqs : numpy.ndarray, optional
        Custom frequencies to use instead of

In [25]:
mother = wlt.Morlet(6.)

In [30]:
print(dc.type)

AttributeError: 'DataFrame' object has no attribute 'type'

In [28]:
wave, scales, freqs, coi, fft, fftfreqs = wlt.cwt(dc['cfs'],
           0.25, 0.25, 0.5, 28, mother)

KeyError: 'ALIGNED'

In [19]:
help(pycwt.wavelet)

Help on module pycwt.wavelet in pycwt:

NAME
    pycwt.wavelet - PyCWT core wavelet transform functions.

FUNCTIONS
    cwt(signal, dt, dj=0.08333333333333333, s0=-1, J=-1, wavelet='morlet', freqs=None)
        Continuous wavelet transform of the signal at specified scales.

        Parameters
        ----------
        signal : numpy.ndarray, list
            Input signal array.
        dt : float
            Sampling interval.
        dj : float, optional
            Spacing between discrete scales. Default value is 1/12.
            Smaller values will result in better scale resolution, but
            slower calculation and plot.
        s0 : float, optional
            Smallest scale of the wavelet. Default value is 2*dt.
        J : float, optional
            Number of scales less one. Scales range from s0 up to
            s0 * 2**(J * dj), which gives a total of (J + 1) scales.
            Default is J = (log2(N * dt / s0)) / dj.
        wavelet : instance of Wavelet class, or

In [7]:
nino3 = pycwt.load_dataset("dc")  # Load sample time-series.

AttributeError: module 'PyCWT' has no attribute 'load_dataset'

In [10]:
wavelet = pycwt.Morlet(6)             # Instantiate mother wavelet.
result = wavelet.run(dc)              # Run wavelet analysis.
result.plot()

AttributeError: 'Morlet' object has no attribute 'run'